# Статистический анализ

## 1. Цели и методы анализа
На этапе EDA были выявлены различия между рядом групп сделок.
Цель данного этапа — оценить величину неопределённости найденных эффектов
и определить, насколько устойчивы наблюдаемые различия при имеющемся
размере выборки.

Основной показатель результативности — математическое ожидание сделки, поскольку оно непосредственно показывает средний финансовый
результат одной сделки в единицах риска. Профит-фактор и доля прибыльных сделок используются как дополнительные описательные
метрики.

### Выбор статистического метода

Распределение `result_r` имеет выраженную асимметрию: большая часть убыточных сделок сосредоточена около −1R, тогда как прибыльные сделки встречаются реже, но имеют значительно больший абсолютный результат +3,8R. Кроме того, размеры сравниваемых групп различаются, а для некоторых выявленных на этапе EDA закономерностей количество наблюдений невелико. Поэтому для оценки неопределённости различий между группами используется бутстрэп-метод (bootstrap). Он позволяет оценить выборочное распределение интересующей статистики без предположения о нормальности исходного распределения.

Для каждой пары групп выполняется 10 000 повторных выборок с возвращением. В каждой итерации из обеих исходных групп формируются бутстрэп-выборки того же размера, после чего рассчитывается разница среднего математического ожидания. На основании полученного распределения разниц определяется 95%-й доверительный интервал через расчет процентилей.

Основной оцениваемый эффект: **ΔE = E(group A) − E(group B)**

Если 95%-й доверительный интервал разницы включает 0, имеющихся данных недостаточно, чтобы уверенно определить направление эффекта. При этом сам наблюдаемый эффект не считается опровергнутым: широкий доверительный интервал указывает прежде всего на высокую неопределённость его оценки.

Бутстрэп не увеличивает фактический размер выборки: 10 000 повторных выборок используются только для оценки неопределённости статистики, а исходный объём данных остаётся равным 130 сделкам.


In [1]:
import numpy as np
import pandas as pd

fact_trades = pd.read_excel('../data/fact_trades_eda.xlsx')

def bootstrap_mean_difference(df, group_col, group_a, group_b, n_bootstrap = 10000, random_state = 42):
    rng = np.random.default_rng(random_state)
    
    sample_a = df.loc[df[group_col] == group_a,'result_r'].dropna().to_numpy()
    sample_b = df.loc[df[group_col] == group_b,'result_r'].dropna().to_numpy()
    
    observed_difference = sample_a.mean() - sample_b.mean()
    bootstrap_differences = []
    
    for _ in range(n_bootstrap):
        bootstrap_a = rng.choice(sample_a, size = len(sample_a),replace = True)
        bootstrap_b = rng.choice(sample_b,size=len(sample_b),replace=True)
        difference = bootstrap_a.mean() - bootstrap_b.mean()
        bootstrap_differences.append(difference)

    ci_low, ci_high = np.percentile(bootstrap_differences,[2.5, 97.5])

    result = pd.DataFrame({
        'group_a': [group_a],
        'group_b': [group_b],
        'n_a': [len(sample_a)],
        'n_b': [len(sample_b)],
        'mean_a': [round(sample_a.mean(), 2)],
        'mean_b': [round(sample_b.mean(), 2)],
        'difference': [round(observed_difference, 2)],
        'ci_low': [round(ci_low, 2)],
        'ci_high': [round(ci_high, 2)]})
    return result

## 2. Проверка устойчивости результатов предварительно заданных гипотез
### 2.1. H1 — влияние рыночной фазы BTC

На этапе EDA математическое ожидание сделок в благоприятных фазах
составило 0.54R против 0.20R в неблагоприятных.

Оценим неопределённость разницы математических ожиданий с помощью
бутстрэп-метода.

In [2]:
btc_bootstrap = bootstrap_mean_difference(fact_trades,'phase_group','favorable','unfavorable')
btc_bootstrap

,group_a,group_b,n_a,n_b,mean_a,mean_b,difference,ci_low,ci_high
0,favorable,unfavorable,86,44,0.54,0.2,0.34,-0.49,1.16


### 2.2. H2 - эффективность повторных входов

На этапе EDA торговые показатели первичных и повторных входов оказались близки: математическое ожидание составило 0.43R для первичных и 0.41R для повторных входов.

Исходная гипотеза заключается не в превосходстве повторных входов над первичными, а в том, что релевантные повторные входы не уступают первичным настолько, чтобы их запрещать.
До расчёта доверительного интервала задано максимально допустимое снижение математического ожидания повторных входов относительно первичных на уровне −0.20R. Эта граница выбрана как практически допустимое ухудшение результативности повторных входов с учётом требований к положительному математическому ожиданию и профит-фактору стратегии.

Рассчитывается разница:**ΔE = E(re-entry) − E(initial)**

Если нижняя граница 95%-го доверительного интервала окажется выше −0.20R, данные будут говорить о достаточной эффективности повторных входов относительно выбранной практической границы. Если интервал пересекает −0.20R, имеющейся выборки недостаточно, чтобы исключить практически неприемлемое ухудшение.

In [3]:
reentry_bootstrap = bootstrap_mean_difference(fact_trades,'entry_attempt','initial','re-entry')
reentry_bootstrap

,group_a,group_b,n_a,n_b,mean_a,mean_b,difference,ci_low,ci_high
0,initial,re-entry,112,18,0.43,0.41,0.02,-1.21,1.15


## 3. Проверка устойчивости результатов закономерностей, выявленных на этапе EDA

### 3.1. Эффективность типа входа spike относительно structure

In [4]:
entry_type_bootstrap = bootstrap_mean_difference(fact_trades,'entry_type','spike','structure')
entry_type_bootstrap

,group_a,group_b,n_a,n_b,mean_a,mean_b,difference,ci_low,ci_high
0,spike,structure,40,90,0.81,0.26,0.55,-0.33,1.46


### 3.2. Эффективность для spike входа относительно уровня above относительно below

In [5]:
entry_vs_level_bootstrap = bootstrap_mean_difference(fact_trades,'entry_vs_level','above','below')
entry_vs_level_bootstrap

,group_a,group_b,n_a,n_b,mean_a,mean_b,difference,ci_low,ci_high
0,above,below,23,16,1.37,0.12,1.24,-0.26,2.69


### 3.3. Эффективность spike вида stepped относительно instant

In [6]:
spike_type_bootstrap = bootstrap_mean_difference(fact_trades,'spike_type','stepped','instant')
spike_type_bootstrap

,group_a,group_b,n_a,n_b,mean_a,mean_b,difference,ci_low,ci_high
0,stepped,instant,20,19,1.41,0.28,1.13,-0.43,2.61


### 3.4. Эффективность рыночной фазы calm_decline относительно остальных

In [7]:
fact_trades ['phase_group_2'] = fact_trades ['btc_phase'].apply (lambda x: 'calm_decline' if x == 'calm_decline' else 'not_calm_decline')
calm_decline_bootstrap = bootstrap_mean_difference(fact_trades,'phase_group_2','calm_decline','not_calm_decline')
calm_decline_bootstrap

,group_a,group_b,n_a,n_b,mean_a,mean_b,difference,ci_low,ci_high
0,calm_decline,not_calm_decline,49,81,0.81,0.2,0.6,-0.21,1.43


## Выводы по статистическому анализу



| Сравнение | Происхождение | N A / N B | ΔE | 95% доверительный интервал | Порог | Результат |
|---|---|---:|---:|---:|---:|---|
| Favorable − Unfavorable | Предварительная H1 | 86 / 44 | +0.34R | [−0.49; +1.16] | 0R | Интервал включает 0 |
| Re-entry − Initial | Предварительная H2 | 18 / 112 | −0.02R | [−1.21; +1.15] | −0.20R | Интервал пересекает порог |
| Spike − Structure | Закономерность | 40 / 90 | +0.55R | [−0.33; +1.46] | 0R | Интервал включает 0 |
| Above − Below | Закономерность | 23 / 16 | +1.24R | [−0.26; +2.69] | 0R | Интервал включает 0 |
| Stepped − Instant | Закономерность | 20 / 19 | +1.13R | [−0.43; +2.61] | 0R | Интервал включает 0 |
| Calm decline − другие фазы | Закономерность | 49 / 81 | +0.60R | [−0.21; +1.43] | 0R | Интервал включает 0 |

Бутстрэп-анализ показал высокую неопределённость оценок всех исследуемых различий. Несмотря на заметные различия математического ожидания, обнаруженные на этапе EDA, 95%-е доверительные интервалы во всех групповых сравнениях оказались широкими и включают нулевую разницу.

Основная причина высокой неопределённости — сочетание ограниченного размера выборки и высокой вариативности результатов сделок `result_r`. Для стратегии характерно асимметричное распределение результатов: большинство убыточных сделок сосредоточено около −1R, тогда как у прибыльных +3,8R. Поэтому присутствие или отсутствие нескольких крупных прибыльных сделок существенно влияет на среднее значение небольших подвыборок.

Наиболее крупные наблюдаемые эффекты получены для входов above относительно below (+1.24R), stepped относительно instant (+1.13R) и фазы calm decline относительно остальных (+0.60R). Однако широкие доверительные интервалы не позволяют надёжно оценить даже направление этих эффектов на текущем объёме данных.

Для H1 наблюдаемое преимущество favorable-фаз составляет +0.34R на сделку, однако 95%-й доверительный интервал [−0.49; +1.16] также включает ноль. Следовательно, текущая выборка не позволяет статистически надёжно подтвердить преимущество агрегированной группы favorable-фаз.

Для H2 используется практическая граница для допустимой эффективности −0.20R вместо нулевой разницы. Наблюдаемая разница между повторными и первичными входами составляет всего −0.02R, однако доверительный интервал пересекает заданную границу. Поэтому текущего количества повторных входов недостаточно, чтобы статистически исключить практически неприемлемое ухудшение их эффективности.

Полученные результаты не отменяют закономерности, обнаруженные на этапе EDA, но показывают, что их величина и даже направление пока оценены недостаточно точно для сильных статистических выводов. Поэтому найденные эффекты следует рассматривать как предварительные кандидаты для проверки на новых данных, а не как доказанные свойства торговой системы.